# 🧠 Sparse Autoencoder (SAE) Playground on Google Colab

This notebook guides you through building a **Sparse Autoencoder (SAE)** from scratch, training it, loading pre-trained weights, and visualizing the semantic concepts it learns.

## 📐 Mathematical Recap
- **Encoder**: Projects centered activations to a sparse latent space:  
  $$f(x) = \text{ReLU}\left((x - b_{dec}) W_{enc} + b_{enc}\right)$$
- **Decoder**: Reconstructs activations from sparse latents:  
  $$\hat{x} = f(x) W_{dec} + b_{dec}$$
- **Loss**: Minimizes MSE while penalizing L1 norm:  
  $$L = \|x - \hat{x}\|_2^2 + \lambda \|f(x)\|_1$$
- **Decoder Normalization**: Normalize rows of $W_{dec}$ (dictionary atoms) to unit L2 norm after each gradient step to prevent the model from cheating the L1 penalty:  
  $$\|W_{dec}[k, :]\|_2 = 1$$

In [ ]:
# 1. Install dependencies
!pip install transformers datasets safetensors tqdm

In [ ]:
# 2. Imports and Device Setup
import os
import math
import random
import json
import html
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.notebook import tqdm
from typing import Generator, List, Dict, Any, Tuple
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download
from IPython.display import display, HTML

# Set up device (Colab T4 GPU uses cuda)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 📦 Pre-made Data Loading & Activation Buffer

In [ ]:
def load_base_model(model_name="gpt2", device="cpu"):
    print(f"Loading base language model: {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
    model.eval()
    for param in model.parameters():
        param.requires_grad = False
    return model, tokenizer

def get_text_dataset(dataset_name="NeelNanda/pile-10k", split="train"):
    print(f"Loading dataset: {dataset_name}...")
    try:
        dataset = load_dataset(dataset_name, split=split)
    except Exception as e:
        print(f"Error loading {dataset_name}: {e}. Falling back to wikitext-2...")
        dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split=split)
        dataset = dataset.filter(lambda x: len(x['text'].strip()) > 0)
    return dataset

class ActivationBuffer:
    def __init__(self, model, tokenizer, dataset, layer_idx, buffer_size=100_000, model_batch_size=16, sae_batch_size=4096, seq_len=128, device="cpu"):
        self.model = model
        self.tokenizer = tokenizer
        self.dataset = dataset
        self.layer_idx = layer_idx
        self.buffer_size = buffer_size
        self.model_batch_size = model_batch_size
        self.sae_batch_size = sae_batch_size
        self.seq_len = seq_len
        self.device = device
        self.d_model = model.config.n_embd
        self.buffer = torch.zeros((buffer_size, self.d_model), dtype=torch.float32, device=device)
        self.pointer = 0
        self.num_filled = 0
        self.dataset_idx = 0
        self.dataset_size = len(dataset)
        self.hook_handle = None
        self.temp_activations = []
        
    def _hook_fn(self, module, input, output):
        if isinstance(output, tuple):
            hidden_states = output[0]
        else:
            hidden_states = output
        self.temp_activations.append(hidden_states.detach().to(self.device))

    def register_hook(self):
        target_layer = self.model.transformer.h[self.layer_idx]
        self.hook_handle = target_layer.register_forward_hook(self._hook_fn)

    def remove_hook(self):
        if self.hook_handle is not None:
            self.hook_handle.remove()
            self.hook_handle = None

    def _fill_buffer(self):
        self.temp_activations = []
        self.register_hook()
        target_vectors = self.buffer_size - self.pointer
        vectors_added = 0
        
        while vectors_added < target_vectors:
            batch_texts = []
            while len(batch_texts) < self.model_batch_size:
                if self.dataset_idx >= self.dataset_size:
                    self.dataset_idx = 0
                item = self.dataset[self.dataset_idx]
                text = item.get("text", "")
                if text.strip():
                    batch_texts.append(text)
                self.dataset_idx += 1
                
            inputs = self.tokenizer(batch_texts, return_tensors="pt", padding="max_length", truncation=True, max_length=self.seq_len).to(self.model.device)
            with torch.no_grad():
                self.model(**inputs)
                
            acts = torch.cat(self.temp_activations, dim=0)
            self.temp_activations = []
            acts_flat = acts.view(-1, self.d_model)
            num_to_copy = min(acts_flat.shape[0], target_vectors - vectors_added)
            
            start_idx = self.pointer
            end_idx = start_idx + num_to_copy
            self.buffer[start_idx:end_idx] = acts_flat[:num_to_copy].to(self.device)
            self.pointer += num_to_copy
            vectors_added += num_to_copy
            
        shuffled_indices = torch.randperm(self.buffer_size, device=self.device)
        self.buffer = self.buffer[shuffled_indices]
        self.pointer = 0
        self.num_filled = self.buffer_size
        self.remove_hook()

    def __iter__(self) -> Generator[torch.Tensor, None, None]:
        while True:
            if self.num_filled - self.pointer < self.sae_batch_size:
                residual_size = self.num_filled - self.pointer
                if residual_size > 0:
                    self.buffer[0:residual_size] = self.buffer[self.pointer:self.num_filled]
                    self.pointer = residual_size
                else:
                    self.pointer = 0
                self._fill_buffer()
                
            start = self.pointer
            end = start + self.sae_batch_size
            self.pointer = end
            yield self.buffer[start:end]

## 🛠️ Implement the Sparse Autoencoder & Trainer Stubs

Fill in the missing mathematical implementation details in the cells below.

In [ ]:
class SparseAutoencoder(nn.Module):
    def __init__(self, d_in: int, d_sae: int, l1_coef: float = 3e-4):
        """
        Sparse Autoencoder (SAE) skeleton.
        """
        super().__init__()
        self.d_in = d_in
        self.d_sae = d_sae
        self.l1_coef = l1_coef
        
        # TODO: Initialize parameters as nn.Parameter
        # - self.W_enc: shape (d_in, d_sae)
        # - self.b_enc: shape (d_sae,)
        # - self.W_dec: shape (d_sae, d_in)
        # - self.b_dec: shape (d_in,)
        
        self.reset_parameters()
        
    def reset_parameters(self):
        # TODO: Initialize weights (e.g. kaiming_uniform_) and biases to zero.
        # Make sure to call self.make_decoder_weights_unit_norm() at the end.
        pass
        
    @torch.no_grad()
    def make_decoder_weights_unit_norm(self):
        # TODO: Normalize each row of self.W_dec (shape: d_sae, d_in) to have L2 norm = 1.0
        # HINT: To avoid leaf variable errors, perform operations on `self.W_dec.data` directly!
        pass
        
    def encode(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Implement encoder projection
        # x_centered = x - b_dec
        # return ReLU(x_centered @ W_enc + b_enc)
        raise NotImplementedError()
        
    def decode(self, feature_acts: torch.Tensor) -> torch.Tensor:
        # TODO: Implement decoder reconstruction
        # return feature_acts @ W_dec + b_dec
        raise NotImplementedError()
        
    def forward(self, x: torch.Tensor):
        # TODO: Implement forward pass returning (x_hat, feature_acts)
        raise NotImplementedError()
        
    def compute_loss(self, x: torch.Tensor, x_hat: torch.Tensor, feature_acts: torch.Tensor):
        # TODO: Implement losses
        # recon_loss = MSE difference
        # sparsity_loss = L1 norm of activations (summed over features, averaged over batch)
        # return (recon_loss + l1_coef * sparsity_loss), recon_loss, sparsity_loss
        raise NotImplementedError()
        
    @torch.no_grad()
    def compute_metrics(self, x: torch.Tensor, x_hat: torch.Tensor, feature_acts: torch.Tensor):
        l0 = (feature_acts > 1e-5).float().sum(dim=-1).mean().item()
        total_var = x.var(dim=0).sum()
        error_var = (x - x_hat).var(dim=0).sum()
        explained_var = (1.0 - error_var / total_var).item() if total_var > 1e-8 else 0.0
        return {"l0": l0, "explained_variance": explained_var}

In [ ]:
class SAETrainer:
    def __init__(self, sae, activation_buffer, lr=3e-4, total_steps=10000, lr_warmup_steps=500, l1_warmup_steps=1000, dead_feature_window=1000, device="cpu"):
        self.sae = sae.to(device)
        self.buffer = activation_buffer
        self.lr = lr
        self.total_steps = total_steps
        self.lr_warmup_steps = lr_warmup_steps
        self.l1_warmup_steps = l1_warmup_steps
        self.dead_feature_window = dead_feature_window
        self.device = device
        self.optimizer = optim.Adam(self.sae.parameters(), lr=self.lr, betas=(0.9, 0.999))
        self.steps_since_active = torch.zeros(self.sae.d_sae, device=self.device)
        self.history = []

    def get_lr_scale(self, step: int) -> float:
        if step < self.lr_warmup_steps:
            return float(step) / float(max(1, self.lr_warmup_steps))
        progress = float(step - self.lr_warmup_steps) / float(max(1, self.total_steps - self.lr_warmup_steps))
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    def get_l1_scale(self, step: int) -> float:
        if step < self.l1_warmup_steps:
            return float(step) / float(max(1, self.l1_warmup_steps))
        return 1.0

    def step_train(self, step: int) -> Dict[str, float]:
        self.optimizer.zero_grad()
        x = next(self.buffer_iter)
        
        # TODO: Implement training step update!
        # 1. Forward pass: x_hat, feature_acts = self.sae(x)
        # 2. Compute loss (with dynamic L1 scale: self.get_l1_scale(step))
        # 3. Backward pass, clip gradients, and step optimizer
        # 4. Normalize W_dec using make_decoder_weights_unit_norm()
        # 5. Track dead features in self.steps_since_active
        
        raise NotImplementedError()

    def train(self):
        self.sae.train()
        self.steps_since_active.zero_()
        self.buffer_iter = iter(self.buffer)
        self.sae.b_dec.data = next(self.buffer_iter).mean(dim=0).clone()
        
        progress_bar = tqdm(range(self.total_steps), desc="SAE Training")
        for step in progress_bar:
            metrics = self.step_train(step)
            self.history.append(metrics)
            
            if step % 100 == 0:
                progress_bar.set_postfix({
                    "loss": f"{metrics['loss']:.4f}",
                    "recon": f"{metrics['recon_loss']:.4f}",
                    "l0": f"{metrics['l0']:.1f}",
                    "dead%": f"{metrics['pct_dead_features']:.1f}%"
                })
        return self.history

## 🧪 Unit Testing Harness

Run this cell to test your class design once implemented.

In [ ]:
def test_sae():
    print("Testing SparseAutoencoder implementation...")
    d_in, d_sae = 768, 3072
    try:
        sae = SparseAutoencoder(d_in=d_in, d_sae=d_sae)
        print("[PASS] Initialization successful.")
    except Exception as e:
        print(f"[FAIL] Initialization: {e}")
        return
        
    params = dict(sae.named_parameters())
    req = {"W_enc": (d_in, d_sae), "b_enc": (d_sae,), "W_dec": (d_sae, d_in), "b_dec": (d_in,)}
    for name, shape in req.items():
        if name not in params:
            print(f"[FAIL] Missing parameter: {name}")
            return
        if tuple(params[name].shape) != shape:
            print(f"[FAIL] Shape mismatch for {name}: {tuple(params[name].shape)} vs expected {shape}")
            return
    print("[PASS] Parameter shapes validated.")
    
    try:
        sae.W_dec.data = torch.randn(d_sae, d_in) * 10.0
        sae.make_decoder_weights_unit_norm()
        norms = sae.W_dec.norm(p=2, dim=1)
        if not torch.allclose(norms, torch.ones(d_sae), atol=1e-4):
            print("[FAIL] Decoder weight normalization has incorrect norms.")
            return
        print("[PASS] unit-norm constraints verified.")
    except Exception as e:
        print(f"[FAIL] Decoder weight normalization: {e}")
        return
        
    x = torch.randn(8, d_in)
    try:
        x_hat, feature_acts = sae(x)
        if tuple(x_hat.shape) != (8, d_in) or tuple(feature_acts.shape) != (8, d_sae):
            print("[FAIL] Forward pass outputs incorrect shapes.")
            return
        if (feature_acts < 0).any():
            print("[FAIL] Hidden activations contain negative values. ReLU missing?")
            return
        print("[PASS] Forward pass validated.")
    except Exception as e:
        print(f"[FAIL] Forward pass: {e}")
        return
        
    try:
        loss, recon_loss, sparsity_loss = sae.compute_loss(x, x_hat, feature_acts)
        if loss.dim() != 0 or recon_loss.dim() != 0 or sparsity_loss.dim() != 0:
            print("[FAIL] Loss terms must be scalar tensors.")
            return
        print("[PASS] Loss functions validated.")
    except Exception as e:
        print(f"[FAIL] Loss calculation: {e}")
        return
        
    print("\nALL TESTS PASSED! Your SparseAutoencoder is ready for training.")

test_sae()

## 🔍 Loading Pre-trained Weights and Visualizing Feature Semantics

In [ ]:
def load_pretrained_sae(layer_idx=6, repo_id="jbloom/GPT2-Small-SAEs-Reformatted", device="cpu"):
    print(f"Downloading pre-trained SAE weights for layer {layer_idx}...")
    folder = f"blocks.{layer_idx}.hook_resid_pre"
    weights_path = hf_hub_download(repo_id=repo_id, filename=f"{folder}/sae_weights.safetensors")
    cfg_path = hf_hub_download(repo_id=repo_id, filename=f"{folder}/cfg.json")
    
    with open(cfg_path, "r") as f:
        cfg = json.load(f)
        
    sae = SparseAutoencoder(d_in=cfg["d_in"], d_sae=cfg["d_sae"], l1_coef=cfg.get("l1_coefficient", 3e-4))
    state_dict = load_file(weights_path)
    sae.W_enc.data = state_dict["W_enc"].to(device)
    sae.b_enc.data = state_dict["b_enc"].to(device)
    sae.W_dec.data = state_dict["W_dec"].to(device)
    sae.b_dec.data = state_dict["b_dec"].to(device)
    return sae

In [ ]:
class FeatureExplorer:
    def __init__(self, sae, model, tokenizer, device="cpu"):
        self.sae = sae.eval().to(device)
        self.model = model.eval()
        self.tokenizer = tokenizer
        self.device = device
        self.hook_handle = None
        self.captured_acts = None
        
    def _register_hook(self, layer_idx):
        def hook_fn(module, input, output):
            self.captured_acts = output[0].detach() if isinstance(output, tuple) else output.detach()
        self.hook_handle = self.model.transformer.h[layer_idx].register_forward_hook(hook_fn)
        
    def _remove_hook(self):
        if self.hook_handle is not None:
            self.hook_handle.remove()
            self.hook_handle = None

    @torch.no_grad()
    def build_database(self, dataset, layer_idx, num_docs=200, seq_len=128, batch_size=16, top_k=10):
        self._register_hook(layer_idx)
        d_sae = self.sae.d_sae
        d_in = self.sae.d_in
        top_values = torch.zeros((d_sae, top_k), device=self.device)
        top_doc_indices = torch.full((d_sae, top_k), -1, dtype=torch.long, device=self.device)
        top_token_indices = torch.full((d_sae, top_k), -1, dtype=torch.long, device=self.device)
        tokenized_docs, raw_docs = [], []
        
        docs = [dataset[i]["text"] for i in range(min(num_docs, len(dataset))) if dataset[i].get("text", "").strip()]
        
        for batch_start in tqdm(range(0, len(docs), batch_size), desc="Building visual feature database"):
            batch_texts = docs[batch_start:batch_start+batch_size]
            inputs = self.tokenizer(batch_texts, return_tensors="pt", padding="max_length", truncation=True, max_length=seq_len).to(self.model.device)
            for item in inputs["input_ids"].cpu().numpy():
                tokenized_docs.append(item)
            self.model(**inputs)
            
            acts = self.captured_acts.to(self.device)
            curr_batch_size = acts.shape[0]
            flat_acts = acts.view(-1, d_in)
            feature_acts = self.sae.encode(flat_acts).view(curr_batch_size, seq_len, d_sae)
            
            for b_idx in range(curr_batch_size):
                doc_idx = batch_start + b_idx
                acts_for_doc = feature_acts[b_idx]
                for feat_idx in range(d_sae):
                    feat_acts = acts_for_doc[:, feat_idx]
                    if feat_acts.max() == 0: continue
                    
                    combined_vals = torch.cat([top_values[feat_idx], feat_acts])
                    new_docs = torch.full((seq_len,), doc_idx, dtype=torch.long, device=self.device)
                    new_tokens = torch.arange(seq_len, dtype=torch.long, device=self.device)
                    combined_docs = torch.cat([top_doc_indices[feat_idx], new_docs])
                    combined_tokens = torch.cat([top_token_indices[feat_idx], new_tokens])
                    
                    sorted_vals, sorted_indices = torch.sort(combined_vals, descending=True)
                    keep = sorted_indices[:top_k]
                    top_values[feat_idx] = sorted_vals[:top_k]
                    top_doc_indices[feat_idx] = combined_docs[keep]
                    top_token_indices[feat_idx] = combined_tokens[keep]
                    
        self._remove_hook()
        self.database = {
            "top_values": top_values.cpu().numpy(),
            "top_doc_indices": top_doc_indices.cpu().numpy(),
            "top_token_indices": top_token_indices.cpu().numpy(),
            "tokenized_docs": tokenized_docs,
            "raw_docs": docs
        }

    def get_feature_summary(self, feature_idx, context_window=5):
        vals = self.database["top_values"][feature_idx]
        docs = self.database["top_doc_indices"][feature_idx]
        tokens = self.database["top_token_indices"][feature_idx]
        tokenized_docs = self.database["tokenized_docs"]
        
        contexts = []
        for i in range(len(vals)):
            val = vals[i]
            if val <= 1e-5: continue
            doc_idx, token_idx = docs[i], tokens[i]
            token_ids = tokenized_docs[doc_idx]
            
            start_idx = max(0, token_idx - context_window)
            end_idx = min(len(token_ids), token_idx + context_window + 1)
            context_tokens = [self.tokenizer.decode([tid]) for tid in token_ids[start_idx:end_idx]]
            contexts.append({
                "activation": float(val),
                "doc_idx": int(doc_idx),
                "token_idx": int(token_idx),
                "active_token": self.tokenizer.decode([token_ids[token_idx]]),
                "context_tokens": context_tokens,
                "highlight_position": token_idx - start_idx
            })
        return contexts

    @torch.no_grad()
    def analyze_custom_text(self, text, layer_idx):
        self._register_hook(layer_idx)
        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)
        input_ids = inputs["input_ids"][0].cpu().numpy()
        tokens = [self.tokenizer.decode([tid]) for tid in input_ids]
        self.model(**inputs)
        self._remove_hook()
        
        acts = self.captured_acts.to(self.device)
        seq_len = acts.shape[1]
        flat_acts = acts.view(-1, self.sae.d_in)
        feature_acts = self.sae.encode(flat_acts)
        
        analysis = []
        for t_idx in range(seq_len):
            t_acts = feature_acts[t_idx]
            active_indices = torch.where(t_acts > 1e-5)[0].cpu().numpy()
            active_vals = t_acts[active_indices].cpu().numpy()
            sort_order = np.argsort(active_vals)[::-1]
            active_indices = active_indices[sort_order]
            active_vals = active_vals[sort_order]
            
            token_features = [{"feature_idx": int(f_idx), "activation": float(val)} for f_idx, val in zip(active_indices, active_vals)]
            analysis.append({"token": tokens[t_idx], "token_idx": t_idx, "features": token_features})
            
        max_acts, max_indices = torch.max(feature_acts, dim=0)
        active_indices = torch.where(max_acts > 1e-5)[0].cpu().numpy()
        active_max_vals = max_acts[active_indices].cpu().numpy()
        active_token_indices = max_indices[active_indices].cpu().numpy()
        sort_order = np.argsort(active_max_vals)[::-1]
        active_indices = active_indices[sort_order]
        active_max_vals = active_max_vals[sort_order]
        active_token_indices = active_token_indices[sort_order]
        
        top_features = [
            {"feature_idx": int(f_idx), "max_activation": float(val), "activating_token": tokens[t_idx]}
            for f_idx, val, t_idx in zip(active_indices, active_max_vals, active_token_indices)
        ]
        return {"tokens": tokens, "analysis": analysis, "top_features": top_features}

In [ ]:
def visualize_feature(explorer, feat_idx, context_window=6):
    contexts = explorer.get_feature_summary(feat_idx, context_window=context_window)
    if not contexts:
        display(HTML(f"<div style='color: #a1a1aa; padding: 10px;'>Feature {feat_idx} did not activate on any tokens.</div>"))
        return
    
    html_str = f"""
    <div style="background-color: #0d0d11; color: #f4f4f7; padding: 20px; border-radius: 12px; border: 1px solid rgba(255,255,255,0.08); font-family: sans-serif;">
        <h3 style="margin-top:0; font-family: sans-serif;">Feature #{feat_idx}</h3>
        <p style="color: #a1a1aa; font-size: 0.9em; margin-bottom: 15px;">Fires primarily on: {', '.join([f\"'{t}'\" for t in set([c['active_token'] for c in contexts])])}</p>
        <div style="display: flex; flex-direction: column; gap: 10px;">
    """
    for ctx in contexts:
        tokens = ctx["context_tokens"]
        hl_pos = ctx["highlight_position"]
        ctx_html = ""
        for i, t in enumerate(tokens):
            escaped_t = html.escape(t)
            if i == hl_pos:
                ctx_html += f"<span style='background-color: rgba(139, 92, 246, 0.25); color: #d8b4fe; padding: 2px 6px; border-radius: 4px; font-weight: bold; border: 1px solid rgba(139, 92, 246, 0.4);'>{escaped_t}</span>"
            else:
                ctx_html += escaped_t
        
        html_str += f"""
            <div style="background: rgba(255,255,255,0.02); padding: 12px; border-radius: 8px; border: 1px solid rgba(255,255,255,0.04);">
                <div style="display: flex; justify-content: space-between; font-size: 0.75em; color: #52525b; margin-bottom: 6px;">
                    <span>Doc #{ctx['doc_idx']} at Pos {ctx['token_idx']}</span>
                    <span style="color: #8b5cf6; font-weight: 500;">Activation: {ctx['activation']:.4f}</span>
                </div>
                <div style="font-size: 0.95em; line-height: 1.6; font-family: monospace;">... {ctx_html} ...</div>
            </div>
        """
    html_str += "</div></div>"
    display(HTML(html_str))

def visualize_text_analysis(explorer, text, layer_idx=6):
    analysis = explorer.analyze_custom_text(text, layer_idx=layer_idx)
    
    chips_html = ""
    for i, item in enumerate(analysis["analysis"]):
        escaped_tok = html.escape(item["token"])
        has_feats = len(item["features"]) > 0
        if has_feats:
            max_act = item["features"][0]["activation"]
            opacity = min(0.8, max_act / 10.0 + 0.15)
            border_color = f"rgba(139, 92, 246, {opacity + 0.2})"
            bg_color = f"rgba(139, 92, 246, {opacity})"
            chips_html += f"""
            <span title=\"Max Act: {max_act:.2f}\" style=\"display: inline-block; background-color: {bg_color}; border: 1px solid {border_color}; color: #d8b4fe; padding: 4px 8px; margin: 4px; border-radius: 6px; font-family: monospace; font-size: 0.9em;\">
                {escaped_tok}
            </span>
            """
        else:
            chips_html += f"""
            <span style=\"display: inline-block; background-color: rgba(255,255,255,0.02); border: 1px solid rgba(255,255,255,0.08); color: #a1a1aa; padding: 4px 8px; margin: 4px; border-radius: 6px; font-family: monospace; font-size: 0.9em;\">
                {escaped_tok}
            </span>
            """
    
    top_feats_html = ""
    for feat in analysis["top_features"][:5]:
        top_feats_html += f"""
        <div style=\"display: flex; justify-content: space-between; background: rgba(255,255,255,0.02); padding: 8px 12px; border-radius: 6px; border: 1px solid rgba(255,255,255,0.04); font-size: 0.9em;\">
            <span><b>Feature #{feat['feature_idx']}</b> on '{feat['activating_token']}'</span>
            <span style=\"color: #06b6d4; font-family: monospace; font-weight: bold;\">{feat['max_activation']:.3f}</span>
        </div>
        """
    
    html_str = f"""
    <div style="background-color: #0d0d11; color: #f4f4f7; padding: 20px; border-radius: 12px; border: 1px solid rgba(255,255,255,0.08); font-family: sans-serif;">
        <h3 style="margin-top:0;">Text Activation Analysis</h3>
        <div style="line-height: 2.2; margin-bottom: 20px; padding: 12px; background: rgba(0,0,0,0.2); border-radius: 8px; border: 1px solid rgba(255,255,255,0.04);">
            {chips_html}
        </div>
        <h4 style="margin-top:0; font-size: 0.95em;">Top Activated Features</h4>
        <div style="display: flex; flex-direction: column; gap: 6px; margin-top: 10px;">
            {top_feats_html}
        </div>
    </div>
    """
    display(HTML(html_str))

## 🏃 Try out the Pre-trained SAE Model

Once your `SparseAutoencoder` class passes tests, execute the cell below to load the model, scan 100 documents to build the activation dataset, and query some features!

In [ ]:
# Load base model
model, tokenizer = load_base_model("gpt2", device=device)

# Load dataset
dataset = get_text_dataset("NeelNanda/pile-10k")

# Load pre-trained SAE for layer 6
sae = load_pretrained_sae(layer_idx=6, device=device)

# Initialize Explorer & scan 150 documents
explorer = FeatureExplorer(sae=sae, model=model, tokenizer=tokenizer, device=device)
explorer.build_database(dataset, layer_idx=6, num_docs=150)

In [ ]:
# Visualize feature activations (e.g. feature index 47, 85, or 105)
visualize_feature(explorer, feat_idx=47)

In [ ]:
# Run a custom text analysis
visualize_text_analysis(explorer, "The president of France spoke at the meeting in Paris about environmental policies.")

## 🚀 Train Your Own Sparse Autoencoder

Once you've verified the code, execute the cell below to initialize the activation buffer and start training your own model from scratch.

In [ ]:
# Initialize Activation Buffer (streams GPT-2 activations on the fly)
buffer = ActivationBuffer(
    model=model,
    tokenizer=tokenizer,
    dataset=dataset,
    layer_idx=6,
    buffer_size=150_000,
    model_batch_size=16,
    sae_batch_size=4096,
    seq_len=128,
    device=device
)

# Instantiate new untrained SAE
my_sae = SparseAutoencoder(d_in=768, d_sae=3072, l1_coef=3e-4)

# Initialize trainer
trainer = SAETrainer(
    sae=my_sae,
    activation_buffer=buffer,
    lr=3e-4,
    total_steps=5000,
    lr_warmup_steps=500,
    l1_warmup_steps=500,
    dead_feature_window=1000,
    device=device
)

# Train model
history = trainer.train()
print("Training complete! Saved weights to checkpoints/my_sae.pt")
os.makedirs("checkpoints", exist_ok=True)
torch.save(my_sae.state_dict(), "checkpoints/my_sae.pt")